# Final Results and Claims

**RESULTS notebook — the locked configuration, dev and eval results, robustness, independent metrics, and the claim ledger**

This is the **results** notebook: it reports the locked configuration and the claim ledger.

Every section follows *Question → What we do → Figure/Table → Reading → Artifact → Caveat*.
Each code cell states what it does and each output is interpreted in the following
cell, so a reader with no access to the code can follow the reasoning. All numbers are
read from immutable artifacts in `results/` through `paper_lib`; missing optional
experiments print `PENDING` with their producer command instead of failing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import paper_lib as L

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 200)
np.random.seed(42)

PRIMARY_RUN = "M4_intersection_dev_conditioned_continuous"
BLIND_RUN = "M4_intersection_dev_blind_continuous"
GATE_FEATURE_COLS = ["top1_faiss", "top5_mean_faiss", "top5_min_faiss", "top5_std_faiss",
                     "retrieval_margin", "top5_spread", "retrieval_overlap", "max_lift",
                     "mean_lift", "n_positive_lifts", "n_negative_lifts", "lift_conflict",
                     "evidence_density", "query_desc_len", "query_title_len"]
print("Project root:", L.ROOT)

## Locked configuration

Chosen on dev before the eval split was touched:

- **Scope:** hierarchical backoff (intersection → team → class → global), minimum evidence 2; fine
  fixed scopes (team, team∩class) as the resolution-informed optimum.
- **Prior:** empirical-Bayes centered Laplace, κ=2, cap ±0.20.
- **Scaling:** pool-relative (λ=0.5) for the ticket-only setting; absolute for resolution-informed.
- **Control policy:** a pre-generation gate over pool-distribution and cosine features, used only
  where harm is concentrated; expected-value (magnitude) modelling when a per-ticket action is needed.

Reported alongside: legacy Laplace fine-scope feedback as the resolution-informed optimum, and the
ticket-only results as the reliability lower bound. All generated dev/eval runs share one generation
regime (model + system prompt + cache); the regenerated dev runs carry an explicit regime id and
warm-cache flag in their summaries.

### What are the generated dev results across all methods and both protocols?

**What we do.** Assemble the canonical dev runs into one table and show the protocol reversal.

**Artifact.** `results/<run>/*_summary.json`

**Caveat.** All rows share one generation regime; eval is reported next.

*What this cell does.* Build the conditioned-vs-blind dev table from the canonical runs and plot it.

In [ ]:
CANON = {
    "M1 global": {"conditioned": "M1_global_dev_conditioned_continuous", "blind": "M1_global_dev_blind_continuous"},
    "M2 team": {"conditioned": "M2_team_dev_conditioned_continuous", "blind": "M2_team_dev_blind_continuous"},
    "M3 class": {"conditioned": "M3_class_dev_conditioned_continuous", "blind": "M3_class_dev_blind_continuous"},
    "M4 intersection": {"conditioned": "M4_intersection_dev_conditioned_continuous", "blind": "M4_intersection_dev_blind_continuous"},
    "M5 backoff (EB)": {"conditioned": "M5_backoff_dev_conditioned_continuous_liftlaplace_eb_pool_std0.5_minev2",
                       "blind": "M5_backoff_dev_blind_continuous_liftlaplace_eb_pool_std0.5_minev2"},
}
rows = []
for method, by_protocol in CANON.items():
    for protocol, folder in by_protocol.items():
        summary = L.load_summary(folder)
        rows.append({"method": method, "protocol": protocol, "n": summary["total_valid"],
                     "mean_delta_cosine": summary["metrics"]["mean_delta_cosine"]})
dev_table = pd.DataFrame(rows)
display(dev_table.pivot(index="method", columns="protocol", values="mean_delta_cosine").round(4))
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(data=dev_table, x="method", y="mean_delta_cosine", hue="protocol",
            palette={"conditioned": "#16856b", "blind": "#c44e52"}, ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Generated-answer cosine delta",
                                           title="Conditioned vs blind (dev, same generation regime)")
plt.tight_layout(); L.savefig("06_dev_protocol_reversal", run_ids=[]); plt.show()

**Reading.** The table is the paper's headline motif, now measured entirely within one generation
regime: resolution-informed fine-scope feedback is positive (team +0.022, intersection +0.031) while
the same methods are negative under ticket-only feedback (team −0.034, intersection −0.051). Broad
scopes (global, class) are negative under both protocols, and the calibrated empirical-Bayes prior is
mildly positive under both (+0.005 conditioned, +0.010 blind). The sign of the effect is controlled by
how the feedback was produced, not by the routing alone.

### Do the results hold on the untouched eval split?

**What we do.** Summaries for the eval runs across protocols and the locked configuration.

**Artifact.** `results registry (eval runs)`

**Caveat.** Eval is reported once; no tuning.

*What this cell does.* Load eval-run summaries from the registry and print them with their headline deltas.

In [ ]:
reg = L.registry()
eval_rows = reg[reg["out_dir"].str.contains("_eval_", na=False)][["script", "out_dir", "headline_metric", "headline_value"]]
display(eval_rows.tail(12).to_string(index=False))

**Reading.** On eval the reversal repeats: resolution-informed fine-scope feedback is positive
(+0.0138 team, +0.0151 intersection; the gated intersection run reaches +0.0164) and ticket-only
feedback is negative (−0.038). The calibrated ticket-only configuration is near zero (+0.002) and the
unseen-procedure (disjoint) variant is mildly positive (+0.005); neither is individually significant.
The eval split confirms the dev conclusions without any tuning.

### Do independent metrics agree?

**What we do.** MiniLM cosine, BGE cosine, ROUGE-L, and BERTScore deltas for the finalists.

**Artifact.** `results/rescored/method_comparison_v2.csv`

**Caveat.** BGE is an independent embedding family.

*What this cell does.* Print the multi-metric table and plot the metric agreement.

In [ ]:
resc = L.load_rescore_comparison()
cols = [c for c in ["run", "delta_cosine_mean", "delta_cosine_bge_mean", "delta_rouge_l_mean", "delta_bertscore_f1_mean"] if c in resc]
display(resc[cols].round(4))
long = resc.melt(id_vars="run", value_vars=[c for c in cols if c != "run"], var_name="metric", value_name="delta")
fig, ax = plt.subplots(figsize=(12, 5)); sns.barplot(data=long, x="run", y="delta", hue="metric", ax=ax)
ax.axhline(0, color="black", lw=1); ax.tick_params(axis="x", rotation=60); ax.set(ylabel="Mean delta", title="Independent-metric agreement")
plt.tight_layout(); L.savefig("06_independent_metrics", run_ids=[]); plt.show()

**Reading.** The independent metrics move in the same direction as the retrieval-family cosine, so
the effects are not an artefact of using the same model family for retrieval and scoring. The gated
resolution-informed intersection run is the strongest independent confirmation: BGE +0.0130 (p=.0002)
and BERTScore +0.0160, with cosine +0.0164 (p=.028). Magnitudes are small throughout; BERTScore is
computed for the eval and gated runs only (dev rows show BGE/ROUGE, computed in the combined rescore).

### Do independent LLM judges agree?

**What we do.** Pairwise answer quality from a non-OpenAI judge (Claude Sonnet 5) on a 100-ticket eval subsample, both orders, win only when consistent.

**Artifact.** `results/answer_judge/{summary.json,scores.csv}`

**Caveat.** Oracle-conditioned judge: it sees the reference reply.

*What this cell does.* Show net win rate, position consistency, and agreement with the cosine sign.

In [ ]:
judge = L.load_answer_judge_summary()
jrows = [{"run": run, **stats} for run, stats in judge["runs"].items()]
jtable = pd.DataFrame(jrows)
display(jtable[["run", "n", "n_identical", "feedback_wins", "baseline_wins",
                "net_win_rate", "position_consistency", "agreement_with_cosine_sign"]].round(3))

**Reading.** The independent judge confirms the direction of the resolution-informed effect
(intersection net win rate +0.31; gated +0.31, i.e. the gate changes almost nothing there) and sees
essentially no difference under calibrated ticket-only feedback (+0.03 ungated; −0.05 gated with 58%
identical answers). Two caveats are reported rather than hidden: the judge flips order in ~25–30% of
pairs, and per-ticket agreement with the embedding-metric sign is low (0.35–0.40) even where the
aggregate direction agrees. The judge is oracle-conditioned (it sees the reference reply), so it is an
upper-bound evaluator, consistent with the conditioned feedback protocol.

### Does the conditioned effect survive a different generator?

**What we do.** Repeat the eval finalists with a non-OpenAI generator (Gemini 3.8 Flash) on the same retrieval and feedback.

**Artifact.** `results/*_gengemini38flash/*_summary.json`

**Caveat.** A different generation regime by design; compared within itself.

*What this cell does.* Compare gemini-generated deltas with the luna runs on the same tickets.

In [ ]:
pairs = [
    ("M2 conditioned", "M2_team_eval_conditioned_continuous", "M2_team_eval_conditioned_continuous_gengemini38flash"),
    ("M4 conditioned", "M4_intersection_eval_conditioned_continuous", "M4_intersection_eval_conditioned_continuous_gengemini38flash"),
    ("M5 blind (calibrated)", "M5_backoff_eval_blind_continuous_liftlaplace_eb_pool_std0.5_minev2",
     "M5_backoff_eval_blind_continuous_liftlaplace_eb_pool_std0.5_minev2_gengemini38flash"),
]
rows = []
for label, luna_folder, gem_folder in pairs:
    luna = L.load_summary(luna_folder)["metrics"]["mean_delta_cosine"]
    gem = L.load_summary(gem_folder)["metrics"]["mean_delta_cosine"]
    rows.append({"config": label, "luna": luna, "gemini-3.8-flash": gem})
cross = pd.DataFrame(rows)
display(cross.round(4))
fig, ax = plt.subplots(figsize=(9, 4.4))
sns.barplot(data=cross.melt(id_vars="config", var_name="generator", value_name="delta"),
            x="config", y="delta", hue="generator", ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Mean generated-answer cosine delta",
                                           title="Cross-generator robustness")
plt.tight_layout(); L.savefig("06_cross_generator", run_ids=[]); plt.show()

**Reading.** The conditioned effect is not a luna artefact: with Gemini 3.8 Flash as the generator
it is larger (M2 +0.029 vs +0.014; M4 +0.040 vs +0.015), and the calibrated blind configuration is
mildly positive (+0.007 vs +0.002). This is a different generation regime by construction (different
model and cache), so it is compared only within itself, and it addresses the same-model
judge/generator tie: the feedback judge never sees generated answers, and a second generator family
reproduces the direction.

### Is the locked configuration robust to seeds and unseen procedures?

**What we do.** Multi-seed dev runs and the disjoint eval run for the locked configuration.

**Artifact.** `results/M5_backoff_*seed*/_summary.json; results/M5_backoff_*disjoint*/_summary.json`

**Caveat.** Feedback source shifts slightly across seeds by construction.

*What this cell does.* Print the seed and disjoint summaries for the calibrated blind winner and the conditioned finalists.

In [ ]:
import json
rows = []
patterns = ["M5_backoff_*seed*", "M5_backoff_*disjoint",
            "M2_team_*seed*", "M2_team_*disjoint", "M4_intersection_*seed*", "M4_intersection_*disjoint"]
for pat in patterns:
    for folder in sorted(L.RESULTS.glob(pat)):
        files = sorted(folder.glob("*_summary.json"), key=lambda p: p.stat().st_mtime)
        if not files:
            continue
        s = json.loads(files[-1].read_text(encoding="utf-8"))
        rows.append({"run": s["experiment_id"], "n": s["total_valid"],
                     "mean_delta_cosine": s["metrics"]["mean_delta_cosine"]})
display(pd.DataFrame(rows).sort_values("run").round(4))

**Reading.** Across seeds the conditioned effect is direction-consistent but smaller than on seed
42: intersection +0.015 (seed 123) and +0.013 (seed 456), team +0.006/−0.001; on the
procedure-disjoint split the conditioned effect persists (intersection +0.015, team +0.010). The
calibrated blind winner is small and mixed in sign across seeds and mildly positive on unseen
procedures. The honest reading is that the resolution-informed effect is the reliable one; the
ticket-only calibrated result is a *de-risking* of feedback rather than a large gain.

### What is the claim ledger?

**What we do.** Every paper claim mapped to the notebook section and artifact that supports it, with readiness.

**Artifact.** `all notebook artifacts`

**Caveat.** Claims marked pending stay conditional.

*What this cell does.* Print the ledger tying each claim to its evidence.

In [ ]:
status = L.artifact_status().set_index("section")["available"].to_dict()
claims = pd.DataFrame([
    ["Feedback benefit is conditional on how it is produced (protocol reversal)", "06 / results", "canonical dev table; eval registry", True],
    ["Fine scopes help; broad pooling harms", "03 / analysis", "retriever_ladder grids", status.get("Granularity: ladder", False)],
    ["Judge scores are zero-inflated; naive Laplace lift saturates and harms under realistic feedback", "04 / modeling", "feedback_calibration/saturation.csv", status.get("Validity: calibration", False)],
    ["Empirical-Bayes centering + pool-relative scaling removes the harm (gains not individually significant)", "04 / modeling", "rescored/method_comparison_v2.csv", status.get("Validity: rescoring", False)],
    ["Lift-formula ordering is stable: EB > tanh > Laplace > LCB", "04 / modeling", "retriever_ladder_liftablation{,_blind}/grid.csv", (L.RESULTS / "retriever_ladder_liftablation" / "grid.csv").exists()],
    ["The gate is risk control: recovers 0.54–0.56 of the ceiling where harm is concentrated, neutral otherwise", "05 / modeling", "gate_study_general/learned_gate{,_decomposition}.csv", (L.RESULTS / "gate_study_general" / "learned_gate_decomposition.csv").exists()],
    ["Expected-value modelling beats sign-only gating; magnitude action selection is positive", "07 / modeling", "magnitude_policy/policy_table.csv", (L.RESULTS / "magnitude_policy" / "policy_table.csv").exists()],
    ["Independent metrics and an independent LLM judge agree in direction on the conditioned effect", "06 / results", "rescored/method_comparison_v2.csv; answer_judge/summary.json", status.get("Pairwise judge", False)],
    ["Negative results: semantic filter, blend null, sign-only multi-action, retriever transfer", "03/07", "ladder grids; multi_action.csv", True],
], columns=["claim", "section", "artifact", "ready"])
display(claims)

## Method card and limitations

**Recommended configuration.** Empirical-Bayes centered Laplace lift, hierarchical backoff
(minimum evidence 2), pool-relative scaling when feedback reliability is unknown; a pre-generation
policy only where harm is concentrated (uncalibrated or unreliable feedback); expected-value
modelling when a per-ticket action must be chosen.

**When it helps.** On moderately uncertain retrievals with trustworthy (resolution-informed)
feedback; the effect reverses when feedback is ticket-only and unreliable, where calibration is the
safer response than gating.

**Limitations.** One organizational corpus (1,595 tickets; dev 319, eval 398); feedback is
LLM-judge-simulated, not human (the conditioned protocol is oracle-informed and simulates an expert
with access to the historical resolution, so it is an upper bound); the judge and the primary
generator are the same model, mitigated only by a second generator in the robustness runs; generated
evidence is concentrated on seed 42, with multi-seed/disjoint coverage for the calibrated blind
winner; ticket-only gains are small and not individually significant; the offline proxy is valid for
configuration selection only; per-ticket agreement between the embedding metric and the LLM judge is
low even when aggregate directions agree.